# MP1 · Prompt Lab — Compare LLM Strategies on a Task

**Starter template.** Fill in the TODOs. ~5-8 hours over 3 days.

Read `learner/MP1_Brief.md` before starting if you haven't already.

---

## Setup

In [2]:
import asyncio
import json
import os
import time
from pathlib import Path

import pandas as pd
from openai import AsyncOpenAI

# Make sure your OPENAI_API_KEY is set in the environment
assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first'

client = AsyncOpenAI()

MODEL = 'gpt-4o-mini'
JUDGE_MODEL = 'gpt-4o'
TEMPERATURE = 0.0

# Cost rates ($ per token) — from W4 cost.py
RATES = {
    'gpt-4o-mini': {'in': 0.15 / 1_000_000, 'out': 0.60 / 1_000_000},
    'gpt-4o':      {'in': 2.50 / 1_000_000, 'out': 10.00 / 1_000_000},
}

print('Setup complete.')

Setup complete.


## Step 1 — Load the data

In [3]:
DATA_DIR = Path('../data')   # adjust if your folder layout differs

snippets = [json.loads(line) for line in (DATA_DIR / 'job_snippets.jsonl').read_text().splitlines() if line.strip()]
golden = {row['id']: row for row in (json.loads(line) for line in (DATA_DIR / 'golden_set.jsonl').read_text().splitlines() if line.strip())}

print(f'Loaded {len(snippets)} snippets, {len(golden)} golden entries.')
print('Sample snippet:', snippets[0])

Loaded 10 snippets, 10 golden entries.
Sample snippet: {'id': 'j01', 'snippet': 'Acme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.'}


## Step 2 — Write the four prompt strategies

Each strategy is a function that takes a snippet text and returns the messages list to send to the LLM.

Implement all four. Keep each one focused — the point is to *see* the difference between strategies, not to over-engineer any one.

**TODO:** fill in the four `prompt_*` functions below.

In [4]:
"""Strategy 1 — zero-shot. Just ask, no examples, no persona."""
# TODO: return a messages list like [{'role': 'user', 'content': '...'}]


def prompt_zero_shot(snippet_text: str) -> list[dict]:

    """Strategy 1 — zero-shot. Just ask, no examples, no persona."""

    return [
        {
            'role': 'user',
            'content': f"""
Extract these three fields from the job posting:

1. company
2. role
3. years_experience_required

Return the result as JSON with exactly these three fields.
years_experience_required must be an integer.

Job posting:
{snippet_text}
"""
        }
    ]

"""Strategy 2 — few-shot. Include 2-3 worked examples in the prompt."""
# TODO: same shape, but include examples
    

def prompt_few_shot(snippet_text: str) -> list[dict]:

    """Strategy 2 — few-shot. Include 2-3 worked examples in the prompt."""

    return [
        {
            'role': 'user',
            'content': f"""
Extract these three fields from the job posting:

1. company
2. role
3. years_experience_required

Return the result as JSON with exactly these three fields.
years_experience_required must be an integer.

Here are some examples:

Example 1:
Job posting: "Acme Corp is looking for a Data Analyst with 3 years of experience."
Answer:
{{"company": "Acme Corp", "role": "Data Analyst", "years_experience_required": 3}}

Example 2:
Job posting: "TechNova is hiring a Senior Python Developer. Candidates should have 5+ years of experience."
Answer:
{{"company": "TechNova", "role": "Senior Python Developer", "years_experience_required": 5}}

Example 3:
Job posting: "Bright Systems needs a Junior Software Engineer with around 2 years of experience."
Answer:
{{"company": "Bright Systems", "role": "Junior Software Engineer", "years_experience_required": 2}}

Now extract the fields from this job posting:

Job posting:
{snippet_text}
"""
        }
    ]


"""Strategy 3 — structured / role-based. Use a system prompt with a persona and explicit JSON schema."""
# TODO: 'You are an expert recruiter... Output JSON with these exact fields...'

def prompt_structured(snippet_text: str) -> list[dict]:

    """Strategy 3 — structured / role-based. Use a system prompt with a persona and explicit JSON schema."""

    return [
        {
            'role': 'system',
            'content': """
You are an expert recruiter who extracts structured information
from job postings.

Your task is to identify exactly these three fields:

{
  "company": "string",
  "role": "string",
  "years_experience_required": "integer"
}

Return only valid JSON with exactly these three fields.
The years_experience_required value must be an integer.
"""
        },
        {
            'role': 'user',
            'content': f"""
Extract the required fields from this job posting:

{snippet_text}
"""
        }
    ]
    

"""Strategy 4 — chain-of-thought. Ask the model to reason before answering."""
# TODO: 'Think step by step, then answer with JSON.'

def prompt_cot(snippet_text: str) -> list[dict]:

    """Strategy 4 — chain-of-thought. Ask the model to reason before answering."""

    return [
        {
            'role': 'user',
            'content': f"""
Extract these three fields from the job posting:

1. company
2. role
3. years_experience_required

Think step by step about the information in the job posting.
Then give the final answer as JSON with exactly these three fields.
years_experience_required must be an integer.

Job posting:
{snippet_text}
"""
        }
    ]


STRATEGIES = {
    'zero_shot': prompt_zero_shot,
    'few_shot': prompt_few_shot,
    'structured': prompt_structured,
    'cot': prompt_cot,
}

In [5]:
test_message = prompt_zero_shot(snippets[0]['snippet'])

print(test_message)

[{'role': 'user', 'content': '\nExtract these three fields from the job posting:\n\n1. company\n2. role\n3. years_experience_required\n\nReturn the result as JSON with exactly these three fields.\nyears_experience_required must be an integer.\n\nJob posting:\nAcme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.\n'}]


In [6]:
test_message = prompt_few_shot(snippets[0]['snippet'])

print(test_message)

[{'role': 'user', 'content': '\nExtract these three fields from the job posting:\n\n1. company\n2. role\n3. years_experience_required\n\nReturn the result as JSON with exactly these three fields.\nyears_experience_required must be an integer.\n\nHere are some examples:\n\nExample 1:\nJob posting: "Acme Corp is looking for a Data Analyst with 3 years of experience."\nAnswer:\n{"company": "Acme Corp", "role": "Data Analyst", "years_experience_required": 3}\n\nExample 2:\nJob posting: "TechNova is hiring a Senior Python Developer. Candidates should have 5+ years of experience."\nAnswer:\n{"company": "TechNova", "role": "Senior Python Developer", "years_experience_required": 5}\n\nExample 3:\nJob posting: "Bright Systems needs a Junior Software Engineer with around 2 years of experience."\nAnswer:\n{"company": "Bright Systems", "role": "Junior Software Engineer", "years_experience_required": 2}\n\nNow extract the fields from this job posting:\n\nJob posting:\nAcme Corp is hiring a Senior S

In [8]:
test_message = prompt_structured(snippets[0]['snippet'])

print(test_message)

[{'role': 'system', 'content': '\nYou are an expert recruiter who extracts structured information\nfrom job postings.\n\nYour task is to identify exactly these three fields:\n\n{\n  "company": "string",\n  "role": "string",\n  "years_experience_required": "integer"\n}\n\nReturn only valid JSON with exactly these three fields.\nThe years_experience_required value must be an integer.\n'}, {'role': 'user', 'content': '\nExtract the required fields from this job posting:\n\nAcme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.\n'}]


In [9]:
test_message = prompt_cot(snippets[0]['snippet'])
print(test_message)

[{'role': 'user', 'content': '\nExtract these three fields from the job posting:\n\n1. company\n2. role\n3. years_experience_required\n\nThink step by step about the information in the job posting.\nThen give the final answer as JSON with exactly these three fields.\nyears_experience_required must be an integer.\n\nJob posting:\nAcme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.\n'}]


In [10]:
print(STRATEGIES.keys())

dict_keys(['zero_shot', 'few_shot', 'structured', 'cot'])


## Step 3 — Async batching

Run all 10 snippets × 4 strategies = 40 calls in parallel.

Capture for each call: strategy, snippet_id, raw response, parsed extraction, cost, latency.

**TODO:** implement `run_one` (single call) and `run_all` (batch all 40).

In [23]:
"""Try to parse a JSON object out of the model's response. Return None if it doesn't parse.
    
Hint: models sometimes wrap JSON in ```json ... ``` fences. Strip them first.
    """
# TODO: extract + parse the JSON, return a dict or None

import json

def parse_response(text: str) -> dict | None:
    """Try to parse a JSON object out of the model's response. Return None if it doesn't parse.
    
    Hint: models sometimes wrap JSON in ```json ... ``` fences. Strip them first.
    """
    text = text.strip()

    if text.startswith("```json"):
        text = text[7:]

    if text.startswith("```"):
        text = text[3:]

    if text.endswith("```"):
        text = text[:-3]

    text = text.strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        return None


"""Run one strategy on one snippet. Return a dict with all the captured fields."""
# TODO: call the model, time the call, compute cost, parse the response

async def run_one(strategy_name: str, snippet: dict) -> dict:
    """Run one strategy on one snippet. Return a dict with all the captured fields."""

    # 1. Create the prompt for this strategy
    messages = STRATEGIES[strategy_name](snippet['snippet'])

    # 2. Start the stopwatch
    start = time.perf_counter()

    # 3. Call the LLM
    response = await client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=TEMPERATURE,
    )

    # 4. Stop the stopwatch
    latency = time.perf_counter() - start

    # 5. Get the model's actual text response
    raw_response = response.choices[0].message.content

    # 6. Convert the JSON text into a Python dictionary
    parsed = parse_response(raw_response)

    # 7. Get token usage
    usage = response.usage

    # 8. Calculate the input and output cost
    input_cost = usage.prompt_tokens * RATES[MODEL]['in']
    output_cost = usage.completion_tokens * RATES[MODEL]['out']
    cost_usd = input_cost + output_cost

    # 9. Return everything we need to record
    return {
        'strategy': strategy_name,
        'snippet_id': snippet['id'],
        'raw_response': raw_response,
        'parsed': parsed,
        'cost_usd': cost_usd,
        'latency': latency,
    }


"""Run all 10 × 4 = 40 calls in parallel. Use asyncio.gather."""
# TODO: build the task list, gather, return results

async def run_all() -> list[dict]:
    """Run all 10 × 4 = 40 calls in parallel. Use asyncio.gather."""

    tasks = []

    for snippet in snippets:
        for strategy_name in STRATEGIES:
            tasks.append(
                run_one(strategy_name, snippet)
            )

    results = await asyncio.gather(*tasks)

    return results


In [12]:
test_json = '{"company": "Acme Corp", "role": "Engineer", "years_experience_required": 5}'

print(parse_response(test_json))

{'company': 'Acme Corp', 'role': 'Engineer', 'years_experience_required': 5}


In [13]:
test_json = '''```json
{"company": "Acme Corp", "role": "Engineer", "years_experience_required": 5}
```'''

print(parse_response(test_json))

{'company': 'Acme Corp', 'role': 'Engineer', 'years_experience_required': 5}


In [15]:
messages = STRATEGIES['zero_shot'](snippets[0]['snippet'])

print(messages)

[{'role': 'user', 'content': '\nExtract these three fields from the job posting:\n\n1. company\n2. role\n3. years_experience_required\n\nReturn the result as JSON with exactly these three fields.\nyears_experience_required must be an integer.\n\nJob posting:\nAcme Corp is hiring a Senior Software Engineer to join our platform team. The ideal candidate has 5+ years of backend development experience and strong skills in Python and distributed systems.\n'}]


In [16]:
response = await client.chat.completions.create(
    model=MODEL,
    messages=messages,
    temperature=TEMPERATURE,
)

print(response.choices[0].message.content)

```json
{
  "company": "Acme Corp",
  "role": "Senior Software Engineer",
  "years_experience_required": 5
}
```


In [17]:
parsed = parse_response(response.choices[0].message.content)

print(parsed)

{'company': 'Acme Corp', 'role': 'Senior Software Engineer', 'years_experience_required': 5}


In [18]:
start = time.perf_counter()

response = await client.chat.completions.create(
    model=MODEL,
    messages=messages,
    temperature=TEMPERATURE,
)

latency = time.perf_counter() - start

print(f"Latency: {latency:.3f} seconds")

Latency: 0.890 seconds


In [19]:
print(response.usage)

CompletionUsage(completion_tokens=34, prompt_tokens=91, total_tokens=125, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))


In [20]:
usage = response.usage

input_cost = usage.prompt_tokens * RATES[MODEL]['in']
output_cost = usage.completion_tokens * RATES[MODEL]['out']

cost_usd = input_cost + output_cost

print(f"Input cost:  ${input_cost:.8f}")
print(f"Output cost: ${output_cost:.8f}")
print(f"Total cost:  ${cost_usd:.8f}")

Input cost:  $0.00001365
Output cost: $0.00002040
Total cost:  $0.00003405


In [21]:
tasks = []

for snippet in snippets:
    for strategy_name in STRATEGIES:
        tasks.append((strategy_name, snippet['id']))

print("Number of tasks:", len(tasks))
print("First 4:", tasks[:4])
print("Last 4:", tasks[-4:])

Number of tasks: 40
First 4: [('zero_shot', 'j01'), ('few_shot', 'j01'), ('structured', 'j01'), ('cot', 'j01')]
Last 4: [('zero_shot', 'j10'), ('few_shot', 'j10'), ('structured', 'j10'), ('cot', 'j10')]


In [24]:
results = await run_all()

print("Number of results:", len(results))

Number of results: 40


In [25]:
print(results[0])

{'strategy': 'zero_shot', 'snippet_id': 'j01', 'raw_response': '```json\n{\n  "company": "Acme Corp",\n  "role": "Senior Software Engineer",\n  "years_experience_required": 5\n}\n```', 'parsed': {'company': 'Acme Corp', 'role': 'Senior Software Engineer', 'years_experience_required': 5}, 'cost_usd': 3.405e-05, 'latency': 1.0405211949982913}


In [ ]:
# Run it
results = await run_all()
print(f'Got {len(results)} results.')
results[0]

## Step 4 — Score against the golden set

Three scores per (strategy × snippet) pair:

1. **accuracy** — how many of 3 fields match (0, 1, 2, or 3)?
2. **parse_success** — did the response parse cleanly?
3. **llm_judge_score** — 1-4 score from gpt-4o-as-judge

**TODO:** implement the three score functions.

In [34]:
"""Compare 3 fields. Case-insensitive, whitespace-trimmed for strings. Return 0, 1, 2, or 3."""
# TODO: count exact matches (with normalisation)

def score_accuracy(extracted: dict | None, gold: dict) -> int:
    """Compare 3 fields. Case-insensitive, whitespace-trimmed for strings. Return 0, 1, 2, or 3."""

    if extracted is None:
        return 0

    fields = [
        'company',
        'role',
        'years_experience_required'
    ]

    score = 0

    for field in fields:
        extracted_value = extracted.get(field)
        gold_value = gold.get(field)

        if isinstance(extracted_value, str) and isinstance(gold_value, str):
            extracted_value = extracted_value.strip().lower()
            gold_value = gold_value.strip().lower()

        if extracted_value == gold_value:
            score += 1

    return score


"""Use gpt-4o as a judge. Return integer 1-4.
   
    Rubric (suggested):
      4 — all three fields correct
      3 — two of three correct, no fabricated data
      2 — one of three correct, or fabricated a field
      1 — none correct or unparsable
    """
    # TODO: prompt the judge with both the gold and the extracted, ask for a 1-4 score


async def score_llm_judge(
    snippet_text: str,
    extracted: dict | None,
    gold: dict
) -> int:
    """Use gpt-4o as a judge. Return integer 1-4."""

    judge_prompt = f"""
You are evaluating an information extraction result.

Job posting:
{snippet_text}

Golden answer:
{json.dumps(gold)}

Extracted answer:
{json.dumps(extracted)}

Use this rubric:

4 — all three fields correct
3 — two of three correct, no fabricated data
2 — one of three correct, or fabricated a field
1 — none correct or unparsable

Return ONLY one integer: 1, 2, 3, or 4.
"""

    response = await client.chat.completions.create(
        model=JUDGE_MODEL,
        messages=[
            {
                'role': 'user',
                'content': judge_prompt
            }
        ],
        temperature=0.0,
    )

    text = response.choices[0].message.content.strip()

    try:
        score = int(text)
        if score in [1, 2, 3, 4]:
            return score
    except ValueError:
        pass

    return 1

In [32]:
score = score_accuracy(
    results[0]['parsed'],
    golden['j01']
)

print("Accuracy:", score, "/ 3")

Accuracy: 3 / 3


In [33]:
for result in results:
    gold = golden[result['snippet_id']]

    result['accuracy'] = score_accuracy(
        result['parsed'],
        gold
    )

    result['parse_success'] = result['parsed'] is not None

print("Scored", len(results), "results.")

Scored 40 results.


In [36]:
# Apply scoring to all 40 results
scored = []

for result in results:
    gold = golden[result['snippet_id']]

    snippet_text = next(
        item['snippet']
        for item in snippets
        if item['id'] == result['snippet_id']
    )

    accuracy = score_accuracy(
        result['parsed'],
        gold
    )

    parse_success = result['parsed'] is not None

    judge_score = await score_llm_judge(
        snippet_text,
        result['parsed'],
        gold
    )

    scored_result = result.copy()
    scored_result['accuracy'] = accuracy
    scored_result['parse_success'] = parse_success
    scored_result['llm_judge_score'] = judge_score

    scored.append(scored_result)

print(f'Scored {len(scored)} results.')

Scored 40 results.


## Step 5 — Build the comparison table

In [38]:
df = pd.DataFrame(scored)

summary = df.groupby('strategy').agg({
    'accuracy': 'mean',
    'parse_success': 'mean',
    'llm_judge_score': 'mean',
    'cost_usd': 'sum',
    'latency': 'median',
}).round(3)

summary.columns = ['Accuracy (mean of 3)', 'Parse rate', 'Judge score', 'Total cost ($)', 'Latency p50 (s)']
summary

,Accuracy (mean of 3),Parse rate,Judge score,Total cost ($),Latency p50 (s)
strategy,,,,,
cot,0.0,0.0,1.0,0.001,2.013
few_shot,2.8,1.0,3.3,0.001,1.024
structured,2.6,1.0,3.2,0.000,1.081
zero_shot,2.6,1.0,3.3,0.000,1.126


## Step 6 — Write your reflection

Open `mp1_writeup.md` and answer the four questions from the brief.

Then commit:

```bash
git add mp1/
git commit -m 'feat(mp1): prompt strategy comparison + writeup'
```

In [40]:
cot_results = df[df['strategy'] == 'cot']

cot_results[['snippet_id', 'raw_response', 'parsed', 'accuracy', 'parse_success']]

,snippet_id,raw_response,parsed,accuracy,parse_success
3,j01,To extract the required fields from the job po...,None,0,False
7,j02,To extract the required fields from the job po...,None,0,False
11,j03,To extract the required fields from the job po...,None,0,False
15,j04,To extract the required fields from the job po...,None,0,False
19,j05,To extract the required fields from the job po...,None,0,False
23,j06,To extract the required fields from the job po...,None,0,False
27,j07,To extract the required fields from the job po...,None,0,False
31,j08,To extract the required fields from the job po...,None,0,False
35,j09,To extract the required fields from the job po...,None,0,False
39,j10,To extract the required fields from the job po...,None,0,False
